# Step 4A.7 — встановлення transformers + завантаження SecBERT для B3
Свідоме, разове встановлення/завантаження (на відміну від Step 4A.6, який нічого не
чіпав). Основний кандидат — `jackaduma/SecBERT` (той самий, що планувався від самого
початку статті). Якщо він не підійде за якістю пізніше — альтернативи: `ehsanaghaei/SecureBERT`
(навчена на CTI-текстах) чи `markusbayer/CySecBERT` (датасет включає CVE-звіти).

Ядро: IDS Research.

In [1]:
import sys, subprocess

def pip_install(pkg):
    print(f"Встановлюю {pkg}...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    if r.returncode != 0:
        print(f"⚠ Помилка встановлення {pkg}:")
        print(r.stderr[-1000:])
    else:
        print(f"✅ {pkg} встановлено")
    return r.returncode == 0

import importlib.util
if importlib.util.find_spec("transformers") is None:
    pip_install("transformers")
else:
    print("transformers вже встановлено")

if importlib.util.find_spec("sentencepiece") is None:
    pip_install("sentencepiece")   # часта залежність токенізаторів, ставимо про всяк випадок

import transformers
print(f"\ntransformers version: {transformers.__version__}")

Встановлюю transformers...
✅ transformers встановлено
Встановлюю sentencepiece...
✅ sentencepiece встановлено

transformers version: 5.17.0


In [2]:
# ---------- Завантаження SecBERT (МЕРЕЖА ДОЗВОЛЕНА, свідоме одноразове завантаження) ----------
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = "jackaduma/SecBERT"
print(f"Завантажую {MODEL_NAME} (перший раз — завантажиться і закешується локально)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"\n✅ Модель і токенізатор завантажено. Пристрій inference: {device}")
print(f"Кількість параметрів: {sum(p.numel() for p in model.parameters()):,}")

Завантажую jackaduma/SecBERT (перший раз — завантажиться і закешується локально)...


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/378k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  336MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Модель і токенізатор завантажено. Пристрій inference: cpu
Кількість параметрів: 83,450,880


/home/ivan_r_opirskyy/ids_research/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
# ---------- Sanity-check inference на реалістичному прикладі CVE description ----------
sample_texts = [
    "Remote code execution vulnerability in the authentication module allows an attacker "
    "to execute arbitrary code via crafted HTTP requests.",
    "",  # порожній опис — перевірка, як модель/токенізатор поводяться з edge case
]

for i, text in enumerate(sample_texts):
    label = "звичайний приклад" if text else "ПОРОЖНІЙ опис (edge case)"
    print(f"\n--- {label} ---")
    inputs = tokenizer(text if text else " ", return_tensors="pt", truncation=True,
                       max_length=256, padding=True).to(device)
    with torch.no_grad():
        output = model(**inputs)
    emb_mean = output.last_hidden_state.mean(dim=1)   # sanity-check pooling, НЕ остаточний вибір
    emb_cls = output.last_hidden_state[:, 0, :]        # CLS-токен, альтернативний варіант
    print(f"  tokens: {inputs['input_ids'].shape[1]}")
    print(f"  embedding dimension: {emb_mean.shape[-1]}")
    print(f"  mean-pooled перші 5 значень: {emb_mean[0, :5].tolist()}")
    print(f"  CLS перші 5 значень:         {emb_cls[0, :5].tolist()}")


--- звичайний приклад ---
  tokens: 22
  embedding dimension: 768
  mean-pooled перші 5 значень: [0.0849284902215004, -0.020695913583040237, 0.2701469957828522, 0.8896159529685974, 0.6184386610984802]
  CLS перші 5 значень:         [0.7724522948265076, -0.5710414052009583, -0.05032116174697876, 1.4940788745880127, 1.333962082862854]

--- ПОРОЖНІЙ опис (edge case) ---
  tokens: 2
  embedding dimension: 768
  mean-pooled перші 5 значень: [0.5987375974655151, 0.1303831934928894, 0.03400737792253494, 0.1663275957107544, -0.10687936842441559]
  CLS перші 5 значень:         [0.5263630747795105, -0.03590989112854004, 0.05951393395662308, 0.34453609585762024, -0.1331639587879181]


In [4]:
# ---------- Швидкий sanity-check: чи модель "розуміє" security-специфічну лексику ----------
# Порівнюємо косинусну близькість для пари схожих (обидва про RCE) і пари різних
# (RCE vs SQL injection) речень — якщо модель адекватна, схожі повинні бути ближчими.
import torch.nn.functional as F

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state.mean(dim=1)

e1 = embed("Remote code execution vulnerability allows arbitrary code execution.")
e2 = embed("This flaw permits an attacker to execute arbitrary commands remotely.")
e3 = embed("SQL injection vulnerability allows unauthorized database access via crafted input.")

sim_similar = F.cosine_similarity(e1, e2).item()
sim_different = F.cosine_similarity(e1, e3).item()
print(f"Косинусна схожість (обидва RCE, мали б бути ближчими): {sim_similar:.4f}")
print(f"Косинусна схожість (RCE vs SQL injection):              {sim_different:.4f}")
if sim_similar > sim_different:
    print("✅ Модель коректно розрізняє: схожі формулювання ближчі, ніж різні теми")
else:
    print("⚠ Несподівано: різні теми виявились ближчими за схожі формулювання — варто перевірити pooling/модель")

Косинусна схожість (обидва RCE, мали б бути ближчими): 0.5649
Косинусна схожість (RCE vs SQL injection):              0.4209
✅ Модель коректно розрізняє: схожі формулювання ближчі, ніж різні теми


In [5]:
# ---------- Підсумок: модель готова, embedding dimension зафіксовано для B3 spec ----------
print("=" * 70)
print(f"ГОТОВО: {MODEL_NAME} завантажено й перевірено")
print("=" * 70)
print(f"Embedding dimension: {emb_mean.shape[-1]}")
print(f"Device: {device}")
print("\nЦе був ЛИШЕ sanity-check (mean vs CLS pooling, порожній опис, схожість речень).")
print("Наступний крок — окрема специфікація B3 encoder:")
print("  - остаточний pooling (mean vs CLS)")
print("  - max_length (наразі 256, перевірити типову довжину CVE description)")
print("  - batch size для масового кодування 379,036 CVE")
print("  - normalize (L2) чи ні")
print("  - обробка порожніх/дуже коротких description")
print("  - кешування embeddings (щоб не перераховувати щоразу)")
print("\nАльтернативи, якщо SecBERT покаже слабкі результати пізніше:")
print("  ehsanaghaei/SecureBERT (навчена на CTI), markusbayer/CySecBERT (датасет включає CVE)")

ГОТОВО: jackaduma/SecBERT завантажено й перевірено
Embedding dimension: 768
Device: cpu

Це був ЛИШЕ sanity-check (mean vs CLS pooling, порожній опис, схожість речень).
Наступний крок — окрема специфікація B3 encoder:
  - остаточний pooling (mean vs CLS)
  - max_length (наразі 256, перевірити типову довжину CVE description)
  - batch size для масового кодування 379,036 CVE
  - normalize (L2) чи ні
  - обробка порожніх/дуже коротких description
  - кешування embeddings (щоб не перераховувати щоразу)

Альтернативи, якщо SecBERT покаже слабкі результати пізніше:
  ehsanaghaei/SecureBERT (навчена на CTI), markusbayer/CySecBERT (датасет включає CVE)
